In [1]:
import torch, transformers
print(torch.__version__, torch.cuda.is_available())
print(transformers.__version__)

2.5.1+cu121 True
5.2.0


In [2]:
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    'D:/models/olmoe-hf',
    dtype=torch.float16,
    device_map='cpu'
)

Loading weights:   0%|          | 0/179 [00:00<?, ?it/s]

In [3]:
gate = model.model.layers[0].mlp.gate
print(gate)

print("\nParamètres :")
for name, param in gate.named_parameters():
    print(f"  {name}: shape={tuple(param.shape)}")

OlmoeTopKRouter()

Paramètres :
  weight: shape=(64, 2048)


In [4]:
import inspect
print(inspect.getsource(gate.forward))

    def forward(self, hidden_states):
        hidden_states = hidden_states.reshape(-1, self.hidden_dim)
        router_logits = F.linear(hidden_states, self.weight)  # (seq_len, num_experts)
        router_logits = torch.nn.functional.softmax(router_logits, dtype=torch.float, dim=-1)
        router_top_value, router_indices = torch.topk(router_logits, self.top_k, dim=-1)  # (seq_len, top_k)
        if self.norm_topk_prob:
            router_top_value /= router_top_value.sum(dim=-1, keepdim=True)
        router_top_value = router_top_value.to(router_logits.dtype)
        router_scores = router_top_value
        return router_logits, router_scores, router_indices



In [5]:
def compute_hobbit_importance(router_scores):
    """
    router_scores: tensor de shape (top_k,) — les probabilités normalisées
                   d'UN SEUL token, déjà triées décroissant par torch.topk.

    Retourne : unimportance_scores, où une valeur PROCHE DE 0 = expert important
               (à garder en haute précision), une valeur PROCHE DE 1 = peu important
               (candidat à la basse précision en cas de cache miss).
    """
    # Équation 2 de HOBBIT : score cumulé des experts classés AVANT celui-ci
    cumulative = torch.cumsum(router_scores, dim=0)
    unimportance = cumulative - router_scores  # exclut le score de l'expert lui-même
    unimportance[0] = 0.0  # le tout premier expert (e0) est toujours jugé important
    return unimportance

# Test avec notre exemple à la main (E3=0.72, E0=0.16, E6=0.12)
exemple_scores = torch.tensor([0.72, 0.16, 0.12])
print(compute_hobbit_importance(exemple_scores))
# Attendu : tensor([0.00, 0.72, 0.88])

tensor([0.0000, 0.7200, 0.8800])


In [6]:
# Un texte d'exemple qu'on va faire passer dans le tokenizer + la première couche
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('D:/models/olmoe-hf')
inputs = tokenizer("Le chat mange une souris", return_tensors="pt")

# On récupère juste l'embedding d'entrée (avant la couche 0) pour simplifier ce premier test
with torch.no_grad():
    hidden = model.model.embed_tokens(inputs['input_ids'])
    router_logits, router_scores, router_indices = model.model.layers[0].mlp.gate(hidden)

print("Scores du premier token :", router_scores[0])
print("Experts sélectionnés :", router_indices[0])
print("Scores d'importance HOBBIT :", compute_hobbit_importance(router_scores[0]))

Scores du premier token : tensor([0.0165, 0.0163, 0.0162, 0.0161, 0.0161, 0.0161, 0.0159, 0.0159])
Experts sélectionnés : tensor([57, 61, 46, 19,  3, 44,  4, 22])
Scores d'importance HOBBIT : tensor([0.0000, 0.0165, 0.0328, 0.0490, 0.0650, 0.0811, 0.0972, 0.1131])


In [7]:
captured = {}

def hook_fn(module, input, output):
    router_logits, router_scores, router_indices = output
    captured['scores'] = router_scores
    captured['indices'] = router_indices

# On attache le hook au routeur de la couche 0
handle = model.model.layers[0].mlp.gate.register_forward_hook(hook_fn)

# Un vrai passage complet dans le modèle (embedding → attention → norm → routeur → ...)
with torch.no_grad():
    model(**inputs)

handle.remove()  # on retire le hook, propreté

print("Scores réels (après le vrai pipeline complet) :", captured['scores'][0])
print("Experts sélectionnés :", captured['indices'][0])
print("Scores d'importance HOBBIT :", compute_hobbit_importance(captured['scores'][0]))

Scores réels (après le vrai pipeline complet) : tensor([0.1346, 0.1062, 0.0921, 0.0855, 0.0829, 0.0705, 0.0682, 0.0214])
Experts sélectionnés : tensor([46, 11, 45, 17, 57, 47,  3, 34])
Scores d'importance HOBBIT : tensor([0.0000, 0.1346, 0.2408, 0.3329, 0.4184, 0.5013, 0.5718, 0.6400])


In [8]:
captured_per_layer = {}

def make_hook(layer_idx):
    def hook_fn(module, input, output):
        _, scores, indices = output
        captured_per_layer[layer_idx] = (scores.detach().clone(), indices.detach().clone())
    return hook_fn

handles = []
for i, layer in enumerate(model.model.layers):
    h = layer.mlp.gate.register_forward_hook(make_hook(i))
    handles.append(h)

with torch.no_grad():
    model(**inputs)

for h in handles:
    h.remove()

print(f"Nombre de couches capturées : {len(captured_per_layer)}")

Nombre de couches capturées : 16


In [9]:
token_idx = 2  # le mot "mange" dans "Le chat mange une souris"

print(f"{'Couche':<8}{'Meilleur score':<16}{'Pire score (top-8)':<20}{'Écart':<10}")
for layer_idx in range(len(captured_per_layer)):
    scores, indices = captured_per_layer[layer_idx]
    token_scores = scores[token_idx]
    best = token_scores[0].item()
    worst = token_scores[-1].item()
    print(f"{layer_idx:<8}{best:<16.4f}{worst:<20.4f}{best - worst:<10.4f}")

Couche  Meilleur score  Pire score (top-8)  Écart     
0       0.1385          0.0238              0.1147    
1       0.0767          0.0235              0.0532    
2       0.0929          0.0244              0.0685    
3       0.1847          0.0226              0.1622    
4       0.0595          0.0247              0.0347    
5       0.0856          0.0231              0.0625    
6       0.0919          0.0216              0.0703    
7       0.0941          0.0229              0.0711    
8       0.1003          0.0289              0.0714    
9       0.0988          0.0391              0.0597    
10      0.1088          0.0266              0.0822    
11      0.1319          0.0378              0.0940    
12      0.1936          0.0311              0.1626    
13      0.2473          0.0221              0.2253    
14      0.1569          0.0341              0.1228    
15      0.2297          0.0224              0.2073    


In [ ]:
sentences = [
    "Le chat mange une souris",
    "Les enfants jouent dans le parc",
    "Elle travaille tous les jours",
    "Le soleil brille sur la montagne",
    "Nous avons visité un musée hier",
    "Il pleut beaucoup cet automne",
    "La voiture roule très vite",
    "Mon frère étudie la physique",
    "Les oiseaux chantent le matin",
    "Ce livre raconte une histoire triste",
    "L'ordinateur a besoin d'une mise à jour",
    "La cuisine sent bon le pain frais",
    "Les étudiants préparent leurs examens",
    "Le train arrive en retard aujourd'hui",
    "Elle a acheté des fleurs pour sa mère",
    "Le chien court après le ballon",
    "Nous devons finir ce projet avant vendredi",
    "La musique adoucit les mœurs, dit-on",
    "Il fait très froid en hiver",
    "Les montagnes sont couvertes de neige",
]

def get_layer_best_scores(sentence, num_layers=16):
    """Retourne le meilleur score du routeur pour chaque couche, sur le dernier
    token de la phrase."""
    inputs = tokenizer(sentence, return_tensors="pt")
    captured = {}

    def make_hook(layer_idx):
        def hook_fn(module, input, output):
            _, scores, indices = output
            captured[layer_idx] = scores.detach().clone()
        return hook_fn

    handles = [
        model.model.layers[i].mlp.gate.register_forward_hook(make_hook(i))
        for i in range(num_layers)
    ]
    with torch.no_grad():
        model(**inputs)
    for h in handles:
        h.remove()

    return [captured[i][-1][0].item() for i in range(num_layers)]

all_results = []
for s in sentences:
    scores = get_layer_best_scores(s)
    all_results.append(scores)
    print(f"{s[:40]:<40} {[f'{x:.3f}' for x in scores]}")

Le chat mange une souris                 ['0.105', '0.083', '0.206', '0.173', '0.107', '0.111', '0.088', '0.085', '0.080', '0.154', '0.080', '0.089', '0.109', '0.109', '0.105', '0.135']
Les enfants jouent dans le parc          ['0.180', '0.154', '0.107', '0.101', '0.120', '0.115', '0.106', '0.081', '0.082', '0.105', '0.162', '0.096', '0.081', '0.104', '0.110', '0.155']
Elle travaille tous les jours            ['0.200', '0.092', '0.179', '0.112', '0.109', '0.162', '0.067', '0.110', '0.074', '0.106', '0.077', '0.098', '0.114', '0.110', '0.101', '0.200']
Le soleil brille sur la montagne         ['0.098', '0.122', '0.104', '0.082', '0.126', '0.078', '0.105', '0.089', '0.099', '0.095', '0.104', '0.093', '0.074', '0.126', '0.097', '0.175']
Nous avons visité un musée hier          ['0.067', '0.048', '0.080', '0.200', '0.087', '0.073', '0.078', '0.098', '0.094', '0.099', '0.096', '0.096', '0.082', '0.095', '0.120', '0.214']
Il pleut beaucoup cet automne            ['0.170', '0.180', '0.059', '

In [ ]:
import numpy as np

all_results = np.array(all_results)  # shape (20 phrases, 16 couches)
mean_per_layer = all_results.mean(axis=0)
std_per_layer = all_results.std(axis=0)

print("\nMoyenne par couche   :", [f"{x:.3f}" for x in mean_per_layer])
print("Écart-type par couche:", [f"{x:.3f}" for x in std_per_layer])


Moyenne par couche   : ['0.143', '0.124', '0.108', '0.114', '0.099', '0.100', '0.088', '0.102', '0.093', '0.094', '0.099', '0.099', '0.097', '0.111', '0.110', '0.160']
Écart-type par couche: ['0.044', '0.038', '0.039', '0.032', '0.019', '0.037', '0.015', '0.019', '0.012', '0.024', '0.020', '0.016', '0.013', '0.011', '0.018', '0.033']


In [ ]:
experts = model.model.layers[0].mlp.experts
import inspect
print(inspect.getsource(experts.forward))


    def forward(
        self,
        hidden_states: torch.Tensor,
        top_k_index: torch.Tensor,
        top_k_weights: torch.Tensor,
    ) -> torch.Tensor:
        final_hidden_states = torch.zeros_like(hidden_states)
        with torch.no_grad():
            expert_mask = torch.nn.functional.one_hot(top_k_index, num_classes=self.num_experts)
            expert_mask = expert_mask.permute(2, 1, 0)
            expert_hit = torch.greater(expert_mask.sum(dim=(-1, -2)), 0).nonzero()

        for expert_idx in expert_hit:
            expert_idx = expert_idx[0]
            if expert_idx == self.num_experts:
                continue
            top_k_pos, token_idx = torch.where(expert_mask[expert_idx])
            current_state = hidden_states[token_idx]
            gate, up = nn.functional.linear(current_state, self.gate_up_proj[expert_idx]).chunk(2, dim=-1)
            current_hidden_states = self.act_fn(gate) * up
            current_hidden_states = nn.functional.linear(current_hidd